# Lesson 2: Maximum-entropy reward maximization

In Lesson 1, maximizing expected reward alone made the Gaussian policy collapse onto a reward maximum. We now introduce a temperature $T$ and fit the policy to the Boltzmann target distribution

$$p_T(x)=\frac{\exp[R(x)/T]}{Z_T}, \qquad T>0,$$

where $T=1/\beta$ and $Z_T$ is the normalizing constant. We will multiply the reverse KL by $T$, producing an objective that also has a direct meaning at $T=0$.

Here $\mathbf{x}\in\mathbb{R}^2$ and the target has four unequally weighted modes. Your task is again to implement two losses: one using reparameterized samples and one using the log-derivative trick.

## Reverse KL and the entropy-regularized objective

For $T>0$, multiply the reverse KL from $q_\theta$ to $p_T$ by the temperature:

$$\begin{aligned}
T\,\mathrm{KL}(q_\theta\,\|\,p_T)
&=T\,\mathbb E_{q_\theta}[\log q_\theta(X)-\log p_T(X)]\\
&=-\mathbb E_{q_\theta}[R(X)]-T\,\mathcal H(q_\theta)+T\log Z_T.
\end{aligned}$$

The term $T\log Z_T$ is constant with respect to $\theta$, so minimizing reverse KL is equivalent to maximizing

$$\boxed{J_T(\theta)=\mathbb E_{q_\theta}[R(X)]+T\,\mathcal H(q_\theta).}$$

Unlike the unscaled KL, this expression can be evaluated directly at zero temperature: $J_0(\theta)=\mathbb E[R(X)]$, which recovers Lesson 1. PyTorch minimizes the loss $-J_T$.

## Setup

Follow `../README.md`, then select the project's `.venv` as the notebook kernel.

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib import animation
import torch
from torch import nn
from torch.distributions import Independent, Normal

torch.manual_seed(7)
plt.rcParams["figure.figsize"] = (7, 4)

## 1. Reward and Boltzmann target

We reuse the weighted 2D multimodal reward from Lesson 1 and choose $T=0.5$ (equivalently $\beta=2$). For visualization and diagnostics, the helper below also handles $T=0$ by showing the limiting point mass at the global reward maximum. Labels $w_i$ are mixture probabilities, not rewards; the gold star marks the global maximum. The training losses do not need the normalizer.

In [ ]:
MODE_LOCATIONS = torch.tensor([[-3.0, -2.0], [-2.0, 2.5], [2.2, 2.7], [3.0, -1.8]])
MODE_WEIGHTS = torch.tensor([0.46, 0.29, 0.17, 0.08])
TARGET_STD = 0.5
TEMPERATURE = 0.5


def target_reward(states: torch.Tensor) -> torch.Tensor:
    """Log-density reward of the weighted 2D Gaussian mixture."""
    standardized = (states.unsqueeze(-2) - MODE_LOCATIONS) / TARGET_STD
    component_log_prob = (
        -0.5 * standardized.square().sum(dim=-1)
        - 2.0 * math.log(TARGET_STD * math.sqrt(2.0 * math.pi))
        + MODE_WEIGHTS.log()
    )
    return torch.logsumexp(component_log_prob, dim=-1)


def boltzmann_target_on_grid(
    rewards: torch.Tensor, x_axis: torch.Tensor, y_axis: torch.Tensor, temperature: float
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return a normalized grid density and T * log(Z_T)."""
    if temperature < 0.0:
        raise ValueError("Temperature must be non-negative.")

    maximum_reward = rewards.max()
    if temperature == 0.0:
        # Plot the limiting point mass as a narrow grid spike.
        maximum_mask = torch.isclose(rewards, maximum_reward).to(rewards.dtype)
        partition = torch.trapezoid(
            torch.trapezoid(maximum_mask, x_axis, dim=1), y_axis, dim=0
        )
        density = maximum_mask / partition
        scaled_log_partition = maximum_reward
    else:
        # Subtract the maximum reward before exponentiating for stability.
        shifted_weights = torch.exp((rewards - maximum_reward) / temperature)
        shifted_partition = torch.trapezoid(
            torch.trapezoid(shifted_weights, x_axis, dim=1), y_axis, dim=0
        )
        density = shifted_weights / shifted_partition
        scaled_log_partition = temperature * shifted_partition.log() + maximum_reward

    return density, scaled_log_partition


plot_axis = torch.linspace(-5.5, 5.5, 250)
grid_x, grid_y = torch.meshgrid(plot_axis, plot_axis, indexing="xy")
grid_points = torch.stack((grid_x, grid_y), dim=-1)
reward_on_grid = target_reward(grid_points)
boltzmann_density, SCALED_LOG_Z_T = boltzmann_target_on_grid(
    reward_on_grid, plot_axis, plot_axis, TEMPERATURE
)
display_reward = reward_on_grid.clamp(min=reward_on_grid.max() - 20)
density_levels = torch.linspace(0.0, boltzmann_density.max().item(), 30)
reward_levels = torch.linspace(display_reward.min().item(), display_reward.max().item(), 30)
GLOBAL_MODE_INDEX = MODE_WEIGHTS.argmax().item()
GLOBAL_MAX_LOCATION = MODE_LOCATIONS[GLOBAL_MODE_INDEX]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)
reward_plot = axes[0].contourf(grid_x, grid_y, display_reward, levels=reward_levels, cmap="YlGn_r")
axes[0].contour(grid_x, grid_y, display_reward, levels=reward_levels[3::4], colors="white", linewidths=0.7, alpha=0.8)
density_plot = axes[1].contourf(grid_x, grid_y, boltzmann_density, levels=density_levels, cmap="YlGn")
axes[1].contour(grid_x, grid_y, boltzmann_density, levels=density_levels[3::4], colors="white", linewidths=0.7, alpha=0.8)
for axis in axes:
    axis.scatter(
        MODE_LOCATIONS[:, 0], MODE_LOCATIONS[:, 1],
        s=700 * MODE_WEIGHTS, c=MODE_WEIGHTS, cmap="viridis",
        edgecolor="white", linewidth=1.2,
    )
    axis.set(xlabel="state x₁", ylabel="state x₂", aspect="equal")
    for mode_index, (location, weight) in enumerate(zip(MODE_LOCATIONS, MODE_WEIGHTS), start=1):
        axis.annotate(f"w{mode_index}={weight.item():.2f}", location + 0.18, color="white", bbox={"facecolor": "black", "alpha": 0.65, "edgecolor": "none", "pad": 1})
    axis.scatter(*GLOBAL_MAX_LOCATION, marker="*", s=280, color="gold", edgecolor="black", linewidth=1.2, zorder=4, label="global max reward")
    axis.legend(loc="lower center", fontsize=8)
axes[0].set_title("Weighted multimodal reward R(x)")
axes[1].set_title(f"Boltzmann target (T={TEMPERATURE:g})")
fig.colorbar(reward_plot, ax=axes[0], label="reward (bottom 20 units clipped)")
fig.colorbar(density_plot, ax=axes[1], label="density")
plt.tight_layout()
plt.show()

print(f"Temperature-scaled log partition T log(Z_T) = {SCALED_LOG_Z_T.item():.4f}")

## 2. Gaussian policy

As before, the network receives a constant observation and parameterizes a diagonal 2D Gaussian. A single Gaussian cannot cover all four separated modes efficiently under reverse KL, so it will normally select one mode. At positive temperature, the entropy term produces finite standard deviations. The network's `log_std` output is not clipped.

In [ ]:
class GaussianPolicy(nn.Module):
    """Neural network that parameterizes a diagonal 2D Gaussian."""

    def __init__(self, hidden_size: int = 32):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(1, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
        )
        self.mean_head = nn.Linear(hidden_size, 2)
        self.log_std_head = nn.Linear(hidden_size, 2)

        # Begin with a broad cloud near the center of the four modes.
        nn.init.zeros_(self.mean_head.weight)
        nn.init.zeros_(self.mean_head.bias)
        nn.init.zeros_(self.log_std_head.weight)
        nn.init.constant_(self.log_std_head.bias, math.log(2.5))

    def forward(self) -> tuple[torch.Tensor, torch.Tensor]:
        device = next(self.parameters()).device
        constant_observation = torch.ones(1, 1, device=device)
        features = self.backbone(constant_observation)

        mean = self.mean_head(features).squeeze(0)
        log_std = self.log_std_head(features).squeeze(0)
        return mean, log_std

    def distribution(self) -> Independent:
        mean, log_std = self()
        return Independent(Normal(mean, log_std.exp()), 1)


policy = GaussianPolicy()
initial_mean, initial_log_std = policy()
print(f"Initial mean: {initial_mean.tolist()}")
print(f"Initial std:  {initial_log_std.exp().tolist()}")

## 3. Loss A: reparameterization estimator

For a Normal policy, the entropy is available analytically from `distribution.entropy()`. Estimate the reward term using differentiable samples

$$X_i=\mu_\theta+\sigma_\theta\varepsilon_i,\qquad \varepsilon_i\sim\mathcal N(0,1),$$

and minimize the negative MaxEnt return

$$L_{\mathrm{path}}=-\frac{1}{N}\sum_i R(X_i)-T\,\mathcal H(q_\theta)=-\widehat J_T.$$

**Task:** use `rsample` for the states and return negative mean reward minus temperature times the policy entropy.

In [ ]:
def reparameterization_maxent_loss(
    policy: GaussianPolicy, num_samples: int, temperature: float = TEMPERATURE
) -> torch.Tensor:
    """Negative pathwise estimate of E[R(X)] + temperature * H(q)."""
    # TODO: Construct the policy distribution.
    # TODO: Draw num_samples differentiable states with rsample.
    # TODO: Compute their mean reward and the distribution entropy.
    # TODO: Return -mean_reward - temperature * entropy.
    raise NotImplementedError("Implement the MaxEnt reparameterization loss")

## 4. Loss B: log-derivative estimator

Write the maximum-entropy reward objective as

$$J_T=\mathbb E_{X\sim q_\theta}[R(X)-T\log q_\theta(X)].$$

Its reward-ascent score signal is $R(X)-T\log q_\theta(X)$. A constant `-T` appears when differentiating the `log q` inside the expectation, but it can be removed because $\mathbb E_q[\nabla_\theta\log q]=0$.

As in Lesson 1, use a leave-one-out baseline. For each sample, subtract the mean learning signal of all other samples. This reduces variance while keeping the estimator unbiased.

**Task:** use non-reparameterized samples, form and detach the baseline-centered reward-ascent signal, then return its negative product with `log_prob` so gradient descent performs ascent.

In [ ]:
def log_derivative_maxent_loss(
    policy: GaussianPolicy, num_samples: int, temperature: float = TEMPERATURE
) -> torch.Tensor:
    """Negative score-function surrogate for maximizing the MaxEnt return."""
    # TODO: Require at least two samples for the leave-one-out baseline.
    # TODO: Construct the distribution and draw samples with sample.
    # TODO: Compute log_prob and signal = reward - temperature * log_prob.
    # TODO: Subtract the leave-one-out mean signal and detach the result.
    # TODO: Return -mean(detached_centered_signal * log_prob).
    raise NotImplementedError("Implement the MaxEnt log-derivative loss")

## 5. Check the implementations

These structural checks verify that each loss is a finite scalar and supplies gradients to all policy parameters.

In [ ]:
def check_loss_function(loss_function) -> None:
    torch.manual_seed(0)
    test_policy = GaussianPolicy()
    loss = loss_function(test_policy, num_samples=128)

    assert loss.ndim == 0, "The loss must be a scalar."
    assert torch.isfinite(loss), "The loss must be finite."
    loss.backward()

    gradients = [parameter.grad for parameter in test_policy.parameters()]
    assert all(gradient is not None for gradient in gradients)
    assert all(torch.isfinite(gradient).all() for gradient in gradients)
    print(f"{loss_function.__name__}: check passed")


check_loss_function(reparameterization_maxent_loss)
check_loss_function(log_derivative_maxent_loss)

## 6. Train both policies

For diagnostics we estimate both the MaxEnt return $J_T$ (which should increase) and $T\,\mathrm{KL}(q\|p_T)$ (which should decrease). At $T=0$, the latter is the reward gap $\max_x R(x)-\mathbb E[R]$. The score-function surrogate value is not itself either diagnostic; only its gradient is useful.

In [ ]:
def policy_statistics(policy: GaussianPolicy) -> tuple[torch.Tensor, torch.Tensor]:
    with torch.no_grad():
        mean, log_std = policy()
    return mean.cpu(), log_std.exp().cpu()


def estimate_temperature_scaled_kl(
    policy: GaussianPolicy, num_samples: int = 4096
) -> float:
    with torch.no_grad():
        distribution = policy.distribution()
        states = distribution.sample((num_samples,))
        scaled_log_density_ratio = (
            -target_reward(states)
            + TEMPERATURE * distribution.log_prob(states)
            + SCALED_LOG_Z_T
        )
    return scaled_log_density_ratio.mean().item()


def estimate_maxent_return(policy: GaussianPolicy, num_samples: int = 4096) -> float:
    with torch.no_grad():
        distribution = policy.distribution()
        mean_reward = target_reward(distribution.sample((num_samples,))).mean()
        return (mean_reward + TEMPERATURE * distribution.entropy()).item()


def train_policy(
    loss_function,
    *,
    seed: int = 7,
    steps: int = 800,
    batch_size: int = 512,
    learning_rate: float = 3e-3,
    log_every: int = 20,
):
    torch.manual_seed(seed)
    policy = GaussianPolicy()
    optimizer = torch.optim.Adam(policy.parameters(), lr=learning_rate)
    # Reusing these noise vectors makes individual dots move smoothly in the GIF.
    animation_noise = torch.randn(160, 2)
    history = {
        "step": [], "mean": [], "std": [],
        "scaled_kl": [], "maxent_return": [], "samples": [],
    }

    for step in range(steps + 1):
        if step % log_every == 0 or step == steps:
            mean, std = policy_statistics(policy)
            history["step"].append(step)
            history["mean"].append(mean)
            history["std"].append(std)
            history["scaled_kl"].append(estimate_temperature_scaled_kl(policy))
            history["maxent_return"].append(estimate_maxent_return(policy))
            history["samples"].append(mean + std * animation_noise)

        if step == steps:
            break

        optimizer.zero_grad()
        loss = loss_function(policy, batch_size)
        loss.backward()
        nn.utils.clip_grad_norm_(policy.parameters(), max_norm=10.0)
        optimizer.step()

    return policy, history

In [ ]:
pathwise_policy, pathwise_history = train_policy(reparameterization_maxent_loss)
score_policy, score_history = train_policy(log_derivative_maxent_loss)

for name, trained_policy in [
    ("Reparameterization", pathwise_policy),
    ("Log derivative", score_policy),
]:
    mean, std = policy_statistics(trained_policy)
    scaled_kl = estimate_temperature_scaled_kl(trained_policy)
    maxent_return = estimate_maxent_return(trained_policy)
    print(
        f"{name:20s} -> mean = {mean.numpy().round(3)}, "
        f"std = {std.numpy().round(3)}, J_T = {maxent_return:.3f}, T*KL = {scaled_kl:.3f}"
    )

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.ravel()

for axis, name, trained_policy, color in [
    (axes[0], "Reparameterization", pathwise_policy, "tab:blue"),
    (axes[1], "Log derivative", score_policy, "tab:orange"),
]:
    axis.contourf(grid_x, grid_y, display_reward, levels=reward_levels, cmap="YlGn_r")
    axis.contour(grid_x, grid_y, display_reward, levels=reward_levels[3::4], colors="white", linewidths=0.7, alpha=0.8)
    axis.scatter(*GLOBAL_MAX_LOCATION, marker="*", s=220, color="gold", edgecolor="black", zorder=4, label="global max reward")
    with torch.no_grad():
        learned_log_density = trained_policy.distribution().log_prob(grid_points)
    peak = learned_log_density.max().item()
    axis.contour(
        grid_x, grid_y, learned_log_density,
        levels=[peak - 4.5, peak - 2.0, peak - 0.5], colors=color, linewidths=2,
    )
    mean, _ = policy_statistics(trained_policy)
    axis.scatter(*mean, marker="*", s=160, color=color, edgecolor="white")
    axis.set(title=f"{name}: final policy", xlabel="state x₁", ylabel="state x₂", aspect="equal")
    axis.legend(loc="lower center", fontsize=8)

axes[2].plot(pathwise_history["step"], pathwise_history["maxent_return"], label="reparameterization")
axes[2].plot(score_history["step"], score_history["maxent_return"], label="log derivative")
axes[2].set(title="Maximum-entropy reward", xlabel="optimization step", ylabel="estimated J_T")
axes[2].legend()

for history, name, color in [
    (pathwise_history, "reparameterization", "tab:blue"),
    (score_history, "log derivative", "tab:orange"),
]:
    std_values = torch.stack(history["std"])
    axes[3].plot(history["step"], std_values[:, 0], color=color, label=f"{name} σ₁")
    axes[3].plot(history["step"], std_values[:, 1], color=color, linestyle="--", label=f"{name} σ₂")
axes[3].axhline(TARGET_STD * math.sqrt(TEMPERATURE), color="black", linestyle=":", label="local target std")
axes[3].set(title="Policy standard deviations", xlabel="optimization step", ylabel="std")
axes[3].legend(fontsize=8)

for axis in axes:
    axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## Discussion

1. Compare the final standard deviations in Lessons 1 and 2. Why does entropy prevent variance collapse while reward is maximized?
2. Why can $T\log Z_T$ be ignored during optimization but not when reporting the temperature-scaled KL value?
3. Reverse KL is mode-seeking. The $w_i$ labels are mixture probabilities, not rewards; how do they and initialization affect which reward maximum is selected?
4. Try $T=0$, $T=0.2$, and $T=2$. How does temperature change the learned variance?
5. Remove the leave-one-out baseline from the score estimator and compare the learning curve.

## 7. Animate the learned samples

Each dot uses the same fixed noise vector in every frame. Its motion therefore shows how the learned mean and standard deviations transform the policy's sample cloud over training. The filled background shows reward from green (low) to yellow (high). Labels $w_i$ are mixture probabilities, and the gold star is the global reward maximum.

In [ ]:
def save_sample_animation(histories, output_path: Path) -> animation.FuncAnimation:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.8), sharex=True, sharey=True)
    sample_artists = []
    trail_artists = []

    for axis, (name, history, color) in zip(axes, histories):
        axis.contourf(grid_x, grid_y, display_reward, levels=reward_levels, cmap="YlGn_r")
        axis.contour(grid_x, grid_y, display_reward, levels=reward_levels[3::4], colors="white", linewidths=0.7, alpha=0.8)
        axis.scatter(
            MODE_LOCATIONS[:, 0], MODE_LOCATIONS[:, 1],
            s=700 * MODE_WEIGHTS, c=MODE_WEIGHTS, cmap="viridis",
            edgecolor="white", linewidth=1.2, zorder=3,
        )
        for mode_index, (location, weight) in enumerate(zip(MODE_LOCATIONS, MODE_WEIGHTS), start=1):
            axis.annotate(f"w{mode_index}={weight.item():.2f}", location + 0.18, color="white", bbox={"facecolor": "black", "alpha": 0.65, "edgecolor": "none", "pad": 1})
        axis.scatter(*GLOBAL_MAX_LOCATION, marker="*", s=260, color="gold", edgecolor="black", linewidth=1.2, zorder=5, label="global max reward")
        axis.legend(loc="lower center", fontsize=8)
        samples = history["samples"][0]
        sample_artists.append(
            axis.scatter(samples[:, 0], samples[:, 1], s=16, alpha=0.55, color=color, zorder=4)
        )
        trail_artists.append(axis.plot([], [], color=color, linewidth=2, zorder=3)[0])
        axis.set(xlim=(-5, 5), ylim=(-5, 5), xlabel="state x₁", ylabel="state x₂", aspect="equal")
        axis.set_title(name)

    step_label = fig.suptitle("")

    def update(frame):
        for (_, history, _), samples_artist, trail_artist in zip(
            histories, sample_artists, trail_artists
        ):
            samples_artist.set_offsets(history["samples"][frame])
            means = torch.stack(history["mean"][: frame + 1])
            trail_artist.set_data(means[:, 0], means[:, 1])
        step_label.set_text(
            f"Maximum-entropy policy samples (T={TEMPERATURE:g}) — "
            f"training step {histories[0][1]['step'][frame]}"
        )
        return [*sample_artists, *trail_artists, step_label]

    sample_animation = animation.FuncAnimation(
        fig, update, frames=len(histories[0][1]["step"]), interval=500,
    )
    sample_animation.save(output_path, writer=animation.PillowWriter(fps=2), dpi=100)
    plt.close(fig)
    return sample_animation


output_directory = Path("L2-MaxEntMin")
if not output_directory.is_dir():
    output_directory = Path(".")
GIF_PATH = output_directory / "maximum_entropy_reward_training.gif"
sample_animation = save_sample_animation(
    [
        ("Reparameterization", pathwise_history, "tab:blue"),
        ("Log derivative", score_history, "tab:orange"),
    ],
    GIF_PATH,
)
print(f"Saved animation to {GIF_PATH.resolve()}")

from IPython.display import Image, display
display(Image(filename=str(GIF_PATH)))